# Bengali LLM Benchmark: TigerLLM-1B-it

State-of-the-art Bengali instruct LLM (1B params, ACL 2025)

**Evaluation**: Zero-shot + 5-shot on full test set (3,553 samples)
**Metrics**: Macro F1 (Type, Target, Severity), CVR, Parse Rate, Latency


In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece
import os
import json
import time
import glob
import re
import gc
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = '/kaggle/working'
RESULTS_DIR = os.path.join(OUTPUT_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    vram = getattr(torch.cuda.get_device_properties(0), 'total_memory', None)
    if vram is not None:
        print(f'VRAM: {vram / 1e9:.1f} GB')


---
## 1. Load Test Data


In [ ]:
# ── Load Data (same split as NB03/NB04) ──
data_files = glob.glob('/kaggle/input/**/train.json', recursive=True)
test_files = glob.glob('/kaggle/input/**/test.json', recursive=True)

if test_files:
    df_test = pd.read_json(test_files[0])
    print(f'Loaded pre-split test set: {test_files[0]} ({len(df_test)} samples)')
elif data_files:
    print('Only train.json found. Splitting to match NB03/NB04...')
    df_all = pd.read_json(data_files[0])
    df_train_split, df_temp = train_test_split(
        df_all, test_size=0.2, random_state=42,
        stratify=df_all['type_of_hate']
    )
    df_dev, df_test = train_test_split(
        df_temp, test_size=0.5, random_state=42,
        stratify=df_temp['type_of_hate']
    )
    print(f'  Train: {len(df_train_split)}, Val: {len(df_dev)}, Test: {len(df_test)}')
else:
    raise FileNotFoundError('No data found!')

# ── Remap labels ──
_LABEL_REMAP = {'Profane': 'Abusive', 'Sexism': 'Gender Hate'}
df_test['type_of_hate'] = df_test['type_of_hate'].fillna('None').astype(str).str.strip().replace(_LABEL_REMAP)
df_test['target_of_hate'] = df_test['target_of_hate'].fillna('None').astype(str).str.strip()
df_test['severity_of_hate'] = df_test['severity_of_hate'].astype(str).str.strip()

# Use FULL test set (3,553 samples)
df_eval = df_test.reset_index(drop=True)
print(f'\nEvaluation samples: {len(df_eval)} (FULL test set)')
print(f'Type distribution:\n{df_eval["type_of_hate"].value_counts().to_string()}')


---
## 2. Shared Utilities


In [ ]:
# ── Constants & Normalization Maps ──
VALID_TYPES = {'None', 'Abusive', 'Political Hate', 'Religious Hate', 'Gender Hate'}
VALID_TARGETS = {'None', 'Individual', 'Organization', 'Community', 'Society'}
VALID_SEVERITIES = {'Little to None', 'Mild', 'Severe'}

TYPE_MAP = {k.lower(): k for k in VALID_TYPES}
TYPE_MAP.update({
    'profane': 'Abusive',
    'sexism': 'Gender Hate',
    'hate': 'Abusive',
    'political': 'Political Hate',
    'religious': 'Religious Hate',
    'gender': 'Gender Hate'
})

TARGET_MAP = {k.lower(): k for k in VALID_TARGETS}
TARGET_MAP.update({
    'individual': 'Individual',
    'organization': 'Organization',
    'community': 'Community',
    'society': 'Society'
})

SEV_MAP = {k.lower(): k for k in VALID_SEVERITIES}
SEV_MAP.update({
    'little to none': 'Little to None',
    'none': 'Little to None',
    'mild': 'Mild',
    'severe': 'Severe',
    'medium': 'Mild',
    'low': 'Little to None',
    'high': 'Severe'
})

def check_consistency_violation(type_pred, target_pred, sev_pred):
    """Returns True if prediction violates consistency rules."""
    if type_pred == 'None':
        if target_pred != 'None' or sev_pred != 'Little to None':
            return True
    return False

def extract_labels_from_json(text):
    """Extract structured labels with primary JSON parser + fallback."""
    # 1. Primary: JSON parse
    try:
        match = re.search(r'\{.*?\}', text, re.DOTALL)
        if match:
            parsed = json.loads(match.group(0))
            return (
                parsed.get('type_of_hate', 'None'),
                parsed.get('target_of_hate', 'None'),
                parsed.get('severity_of_hate', 'Little to None')
            ), True
    except:
        pass
    
    # 2. Fallback: line-by-line key extraction
    try:
        t_val, trg_val, s_val = None, None, None
        for line in text.splitlines():
            line_c = line.strip().replace('"', '').replace("'", '')
            if 'type_of_hate' in line_c:
                t_val = line_c.split(':')[-1].split('=')[-1].strip(' ,{}')
            if 'target_of_hate' in line_c:
                trg_val = line_c.split(':')[-1].split('=')[-1].strip(' ,{}')
            if 'severity_of_hate' in line_c:
                s_val = line_c.split(':')[-1].split('=')[-1].strip(' ,{}')
        if t_val or trg_val or s_val:
            return (
                t_val if t_val else 'None',
                trg_val if trg_val else 'None',
                s_val if s_val else 'Little to None'
            ), True
    except:
        pass
        
    return ('None', 'None', 'Little to None'), False

def normalize_prediction(t, trg, s):
    """Normalize casing and aliases to canonical taxonomy."""
    t_str = str(t).strip().lower()
    trg_str = str(trg).strip().lower()
    s_str = str(s).strip().lower()
    
    t_norm = TYPE_MAP.get(t_str, 'None')
    trg_norm = TARGET_MAP.get(trg_str, 'None')
    s_norm = SEV_MAP.get(s_str, 'Little to None')
    return t_norm, trg_norm, s_norm

# ── Prompt Templates (Pure String Concatenation: 100% bug-proof) ──
PROMPT_SUFFIX = """

শুধুমাত্র নিচের JSON ফরম্যাটে উত্তর দিন:
{"type_of_hate": "...", "target_of_hate": "...", "severity_of_hate": "..."}"""

ZERO_SHOT_PREFIX = """নিচের মন্তব্যটি বিশ্লেষণ করুন এবং এর 'Hate Type', 'Target', এবং 'Severity' নির্ধারণ করুন।

অপশনসমূহ:
Hate Type: None, Abusive, Political Hate, Religious Hate, Gender Hate
Target: None, Individual, Organization, Community, Society
Severity: Little to None, Mild, Severe

মন্তব্য: """

FEW_SHOT_EXAMPLES = [
    {
        "comment": "মেসি বোলে কথা মেসি মেসি মেসি এবং মেসি কিছু বলতে হবে না",
        "type": "None", "target": "None", "severity": "Little to None"
    },
    {
        "comment": "সবাই বয়কট করুন সময় টিভি",
        "type": "Abusive", "target": "Organization", "severity": "Mild"
    },
    {
        "comment": "একটা পণ্য না ভাই কোন পণ্যর দাম কম বলব এই সরকার কি আর বাজার নিয়ন্ত্রণ করতে পারবে না ডাকাতি করবে",
        "type": "Political Hate", "target": "Society", "severity": "Mild"
    },
    {
        "comment": "সরকারের উচিত বাংলাদেশেও মন্দির ভেঙে মসজিদ করা",
        "type": "Religious Hate", "target": "Organization", "severity": "Little to None"
    },
    {
        "comment": "আন্দোলনটা ছিল পতিতা মহিলাদের জন্য",
        "type": "Gender Hate", "target": "Society", "severity": "Severe"
    }
]

few_shot_examples_text = ""
for ex in FEW_SHOT_EXAMPLES:
    c = ex['comment']
    t = ex['type']
    trg = ex['target']
    s = ex['severity']
    few_shot_examples_text += f'\nমন্তব্য: "{c}"\nউত্তর: {{"type_of_hate": "{t}", "target_of_hate": "{trg}", "severity_of_hate": "{s}"}}\n'

FEW_SHOT_PREFIX = """নিচের মন্তব্যটি বিশ্লেষণ করুন এবং এর 'Hate Type', 'Target', এবং 'Severity' নির্ধারণ করুন।

অপশনসমূহ:
Hate Type: None, Abusive, Political Hate, Religious Hate, Gender Hate
Target: None, Individual, Organization, Community, Society
Severity: Little to None, Mild, Severe

এখানে কিছু উদাহরণ দেওয়া হলো:""" + few_shot_examples_text + """
এখন এই মন্তব্যটি বিশ্লেষণ করুন:
মন্তব্য: """

def build_zero_shot_prompt(comment):
    return ZERO_SHOT_PREFIX + '"' + str(comment) + '"' + PROMPT_SUFFIX

def build_few_shot_prompt(comment):
    return FEW_SHOT_PREFIX + '"' + str(comment) + '"' + PROMPT_SUFFIX

def run_benchmark(model, tokenizer, model_name, prompt_fn, max_new_tokens=64):
    """Run benchmark on a loaded model with robust generation and memory management."""
    preds_type, preds_target, preds_sev = [], [], []
    violations = 0
    parse_successes = 0
    start_time = time.time()

    # Determine safe pad token id
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    if pad_id is None:
        pad_id = 0

    for idx, (_, row) in enumerate(tqdm(df_eval.iterrows(), total=len(df_eval), desc=f'{model_name}')):
        prompt = prompt_fn(row['comment'])
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.1,
                do_sample=True,
                pad_token_id=pad_id
            )

        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
        (t, trg, s), parsed = extract_labels_from_json(response)
        t, trg, s = normalize_prediction(t, trg, s)

        if parsed:
            parse_successes += 1
        if check_consistency_violation(t, trg, s):
            violations += 1

        preds_type.append(t)
        preds_target.append(trg)
        preds_sev.append(s)

        # Clear VRAM cache every 500 samples to prevent fragmentation
        if (idx + 1) % 500 == 0:
            torch.cuda.empty_cache()

    elapsed = time.time() - start_time
    latency = elapsed / len(df_eval)
    parse_rate = parse_successes / len(df_eval) * 100

    return preds_type, preds_target, preds_sev, violations, latency, parse_rate

print('Utilities & Prompt builders loaded.')


---
## 3. Benchmark: TigerLLM-1B-it
Zero-shot evaluation first, then 5-shot. Results saved after each pass.


In [ ]:
MODEL_ID = "md-nishat-008/TigerLLM-1B-it"
print(f"Loading {MODEL_ID}...")
tokenizer_tiger = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model_tiger = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
)
model_tiger.eval()

true_type = df_eval['type_of_hate'].tolist()
true_target = df_eval['target_of_hate'].tolist()
true_sev = df_eval['severity_of_hate'].tolist()

def calc_metrics(p_type, p_target, p_sev, viols, latency, parse_rate, name):
    type_f1 = f1_score(true_type, p_type, average='macro', zero_division=0)
    target_f1 = f1_score(true_target, p_target, average='macro', zero_division=0)
    sev_f1 = f1_score(true_sev, p_sev, average='macro', zero_division=0)
    avg_f1 = (type_f1 + target_f1 + sev_f1) / 3
    cvr = viols / len(p_type) * 100
    print(f"  Type F1:       {type_f1:.4f}")
    print(f"  Target F1:     {target_f1:.4f}")
    print(f"  Severity F1:   {sev_f1:.4f}")
    print(f"  Avg Macro F1:  {avg_f1:.4f}")
    print(f"  CVR:           {cvr:.2f}% ({viols}/{len(p_type)})")
    print(f"  Parse Rate:    {parse_rate:.1f}%")
    print(f"  Latency:       {latency:.2f}s/sample")
    return {
        'model': name, 'type_f1': round(type_f1, 4), 'target_f1': round(target_f1, 4),
        'sev_f1': round(sev_f1, 4), 'avg_f1': round(avg_f1, 4), 'cvr_pct': round(cvr, 2),
        'violations': viols, 'total_samples': len(p_type),
        'latency_s': round(latency, 3), 'parse_rate_pct': round(parse_rate, 1)
    }

# Check for existing results (skip if already computed to save hours of GPU time!)
zero_shot_file = os.path.join(RESULTS_DIR, 'tiger_0shot_results.json')
input_zero_shot = glob.glob('/kaggle/input/**/tiger_0shot_results.json', recursive=True)

# ── PASS 1: Zero-Shot ──
if os.path.exists(zero_shot_file):
    print(f"\n✅ Found existing Zero-Shot results at {zero_shot_file}! Skipping Pass 1...")
    with open(zero_shot_file, 'r') as f:
        r_0shot = json.load(f)
elif input_zero_shot:
    print(f"\n✅ Found existing Zero-Shot results in input at {input_zero_shot[0]}! Loading & skipping Pass 1...")
    with open(input_zero_shot[0], 'r') as f:
        r_0shot = json.load(f)
    with open(zero_shot_file, 'w') as f:
        json.dump(r_0shot, f, indent=2)
else:
    print(f"\n============================================================")
    print(f"  ZERO-SHOT EVALUATION: TigerLLM-1B-it")
    print(f"============================================================")
    pt_0, ptr_0, ps_0, v_0, lat_0, pr_0 = run_benchmark(
        model_tiger, tokenizer_tiger, "TigerLLM-1B-it (0-shot)", build_zero_shot_prompt
    )
    r_0shot = calc_metrics(pt_0, ptr_0, ps_0, v_0, lat_0, pr_0, "TigerLLM-1B-it (0-shot)")
    with open(zero_shot_file, 'w') as f:
        json.dump(r_0shot, f, indent=2)
    print(f"\n✅ Zero-shot results saved.")

# Check for existing 5-shot results
five_shot_file = os.path.join(RESULTS_DIR, 'tiger_5shot_results.json')
input_five_shot = glob.glob('/kaggle/input/**/tiger_5shot_results.json', recursive=True)

# ── PASS 2: 5-Shot ──
if os.path.exists(five_shot_file):
    print(f"\n✅ Found existing 5-Shot results at {five_shot_file}! Skipping Pass 2...")
    with open(five_shot_file, 'r') as f:
        r_5shot = json.load(f)
elif input_five_shot:
    print(f"\n✅ Found existing 5-Shot results in input at {input_five_shot[0]}! Loading & skipping Pass 2...")
    with open(input_five_shot[0], 'r') as f:
        r_5shot = json.load(f)
    with open(five_shot_file, 'w') as f:
        json.dump(r_5shot, f, indent=2)
else:
    print(f"\n============================================================")
    print(f"  5-SHOT EVALUATION: TigerLLM-1B-it")
    print(f"============================================================")
    pt_5, ptr_5, ps_5, v_5, lat_5, pr_5 = run_benchmark(
        model_tiger, tokenizer_tiger, "TigerLLM-1B-it (5-shot)", build_few_shot_prompt
    )
    r_5shot = calc_metrics(pt_5, ptr_5, ps_5, v_5, lat_5, pr_5, "TigerLLM-1B-it (5-shot)")
    with open(five_shot_file, 'w') as f:
        json.dump(r_5shot, f, indent=2)
    print(f"\n✅ 5-shot results saved.")

# Free VRAM
del model_tiger, tokenizer_tiger
gc.collect()
torch.cuda.empty_cache()


---
## 4. Comparison with Our Model


In [ ]:
# ── Final Comparison ──
our_result = {
    'model': 'Ours (exp4, 110M)',
    'type_f1': 0.4994, 'target_f1': 0.5598, 'sev_f1': 0.6136,
    'avg_f1': 0.5576, 'cvr_pct': 0.03, 'violations': 1,
    'total_samples': 3553, 'latency_s': 0.015, 'parse_rate_pct': 100.0
}

# Load saved results
with open(os.path.join(RESULTS_DIR, 'tiger_0shot_results.json'), 'r') as f:
    r_0shot = json.load(f)

try:
    with open(os.path.join(RESULTS_DIR, 'tiger_5shot_results.json'), 'r') as f:
        r_5shot = json.load(f)
    has_5shot = True
except:
    has_5shot = False

# Print final comparison table
print(f"\n\n==========================================================================================")
print(f"  FINAL COMPARISON: TigerLLM-1B-it vs Our Model")
print(f"==========================================================================================")
print(f"  {'Model':<35} {'Params':>7} {'Avg F1':>8} {'CVR':>8} {'Parse%':>8} {'Lat(s)':>8}")
print(f"  {'-'*35} {'-'*7} {'-'*8} {'-'*8} {'-'*8} {'-'*8}")
print(f"  {r_0shot['model']:<35} {'1-3B':>7} {r_0shot['avg_f1']:>8.4f} {r_0shot['cvr_pct']:>7.2f}% {r_0shot['parse_rate_pct']:>7.1f}% {r_0shot['latency_s']:>7.3f}s")
if has_5shot:
    print(f"  {r_5shot['model']:<35} {'1-3B':>7} {r_5shot['avg_f1']:>8.4f} {r_5shot['cvr_pct']:>7.2f}% {r_5shot['parse_rate_pct']:>7.1f}% {r_5shot['latency_s']:>7.3f}s")
print(f"  {'-'*35} {'-'*7} {'-'*8} {'-'*8} {'-'*8} {'-'*8}")
print(f"  {'Ours (exp4, 110M)':<35} {'110M':>7} {our_result['avg_f1']:>8.4f} {our_result['cvr_pct']:>7.2f}% {our_result['parse_rate_pct']:>7.1f}% {our_result['latency_s']:>7.3f}s")
print(f"==========================================================================================")

# Save all results together
all_results = [r_0shot]
if has_5shot:
    all_results.append(r_5shot)
all_results.append(our_result)
with open(os.path.join(RESULTS_DIR, 'tiger_full_comparison.json'), 'w') as f:
    json.dump(all_results, f, indent=2)
print(f"\n✅ All results saved to {RESULTS_DIR}/tiger_full_comparison.json")
